# MBPP OOD + TACO Robustness Evaluation

This notebook clones the repo into Colab, installs the evaluation package, discovers or selects Hugging Face models, and lets you run separate evaluation passes for:
- MBPP OOD only
- TACO noisy robustness only
- TACO keyword-replacement probes only
- combined runs if you want everything together


In [ ]:
GIT_REPO_URL = "https://github.com/samuelleecong/CS4248_ez_A.git"
GIT_BRANCH = "ood_perturb"
REPO_DIR = "/content/CS4248_ez_A"

GITHUB_TOKEN = ""  # or use Colab Secret named GITHUB_TOKEN

import os
from urllib.parse import urlparse
import getpass

try:
    from google.colab import userdata  # type: ignore
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass

if not GITHUB_TOKEN:
    GITHUB_TOKEN = getpass.getpass("Enter github token")

%cd /content

parsed = urlparse(GIT_REPO_URL)
auth_repo_url = f"https://oauth2:{GITHUB_TOKEN}@{parsed.netloc}{parsed.path}"

!rm -rf "$REPO_DIR"
!git clone --branch "$GIT_BRANCH" "$auth_repo_url" "$REPO_DIR"

%cd $REPO_DIR
!python -m pip install -q -U pip
!python -m pip install -q textattack
!python -m pip install -q nltk
!python -m pip install -q -e ./mbpp_kd_suite

In [ ]:
# Optional Hugging Face discovery/auth setup.
# Default notebook configuration uses the six saturated TinyBERT checkpoints below.
# Set AUTO_DISCOVER_MODELS=True only if you explicitly want to override MODEL_IDS.
HF_OWNER = "cs4248-nlp"
HF_TOKEN = ""
AUTO_DISCOVER_MODELS = False
MODEL_NAME_FILTER = "paper-s7"

import os

try:
    from google.colab import userdata  # type: ignore
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass("Enter HF Token")
except Exception:
    pass

if HF_TOKEN:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
from huggingface_hub import HfApi

discovered_model_ids = []
if AUTO_DISCOVER_MODELS:
    if not HF_OWNER:
        raise ValueError("Set HF_OWNER when AUTO_DISCOVER_MODELS=True.")
    api = HfApi(token=HF_TOKEN or None)
    discovered = list(api.list_models(author=HF_OWNER, full=True))
    discovered_model_ids = [model.id for model in discovered if model.id]
    if MODEL_NAME_FILTER:
        discovered_model_ids = [model_id for model_id in discovered_model_ids if MODEL_NAME_FILTER.lower() in model_id.lower()]
    discovered_model_ids = sorted(discovered_model_ids)
    print("Discovered models:")
    for model_id in discovered_model_ids:
        print(" -", model_id)
else:
    print("AUTO_DISCOVER_MODELS=False; skipping Hugging Face repo discovery.")

In [ ]:
MODEL_IDS = discovered_model_ids or [
    "cs4248-nlp/paper-s7-control-bs32-tinybert-general-4l-312d-taco-hf-20260402-015143",
    "cs4248-nlp/paper-s7-embed-dw100-aw10-tinybert-general-4l-312d-taco-hf-20260402-015143",
    "cs4248-nlp/paper-s8-a2-bimga-uniform-tinybert-general-4l-312d-taco-hf-20260402-015143",
    "cs4248-nlp/paper-s8-hnp-dw100-pw10-tinybert-general-4l-312d-taco-hf-20260402-015143",
    "cs4248-nlp/paper-s9-bimga-dw50-aw10-tinybert-general-4l-312d-taco-hf-20260402-015143",
    "cs4248-nlp/paper-s9-score-dw100-tinybert-general-4l-312d-taco-hf-20260402-015143",
]

MBPP_DATASET_PATH = None
TACO_DATASET_NAME = "BEE-spoke-data/TACO-hf"
TACO_DATASET_PATH = None
SPLIT = "test"
SPLIT_SEED = 42
BATCH_SIZE = 8
OUTPUT_DIR = "./colab_runs/ood_analysis"
LEXICAL_MAP_PATH = None  # optional override; default uses eval/ood_analysis/lexical_replacements.json

assert MODEL_IDS, "Add at least one Hugging Face model ID to MODEL_IDS."

In [ ]:
# Structural validation: keep flat HF repos and projected student repos.
from huggingface_hub import list_repo_files
import pandas as pd

valid_model_ids = []
invalid_model_rows = []

for model_id in MODEL_IDS:
    try:
        repo_files = set(list_repo_files(model_id, token=HF_TOKEN or None))
        is_flat_hf = "config.json" in repo_files and any(name in repo_files for name in ("model.safetensors", "pytorch_model.bin"))
        is_nested_suite = "backbone/config.json" in repo_files and any(path.startswith("tokenizer/") for path in repo_files)
        if is_flat_hf or is_nested_suite:
            valid_model_ids.append(model_id)
            print("VALID:", model_id)
        else:
            invalid_model_rows.append({"model_id": model_id, "error": "No supported flat HF or nested suite layout found"})
            print("SKIP :", model_id)
            print("ERROR:", "No supported flat HF or nested suite layout found", "\n")
    except Exception as exc:
        invalid_model_rows.append({"model_id": model_id, "error": str(exc)})
        print("SKIP :", model_id)
        print("ERROR:", str(exc)[:250], "\n")

MODEL_IDS = valid_model_ids
print(f"Kept {len(MODEL_IDS)} valid models, skipped {len(invalid_model_rows)}.")
assert MODEL_IDS, "No compatible HF models found after validation."

invalid_df = pd.DataFrame(invalid_model_rows)
invalid_df

## Run Helpers

The helper below lets each run cell specify only the task and perturbation tier it wants.

In [ ]:
import re
import shlex
import shutil
from pathlib import Path

def build_eval_cmd(task, perturbation_tier, output_subdir, lexical_map_path=None):
    model_args = " ".join(f"--model {shlex.quote(model_id)}" for model_id in MODEL_IDS)
    mbpp_arg = "" if MBPP_DATASET_PATH is None else f" --mbpp-dataset-path {shlex.quote(MBPP_DATASET_PATH)}"
    taco_path_arg = "" if TACO_DATASET_PATH is None else f" --taco-dataset-path {shlex.quote(TACO_DATASET_PATH)}"
    lexical_arg = "" if lexical_map_path is None else f" --lexical-map-path {shlex.quote(lexical_map_path)}"
    full_output_dir = str(Path(OUTPUT_DIR) / output_subdir)
    cmd = (
        f"PYTHONPATH=/content/CS4248_ez_A/mbpp_kd_suite:/content/CS4248_ez_A/mbpp_kd_suite/src "
        f"python -m eval.ood_analysis.ood_robustness {model_args}"
        f" --task {shlex.quote(task)}"
        f" --taco-dataset-name {shlex.quote(TACO_DATASET_NAME)}"
        f" --split {shlex.quote(SPLIT)}"
        f" --split-seed {SPLIT_SEED}"
        f" --perturbation-tier {shlex.quote(perturbation_tier)}"
        f" --batch-size {BATCH_SIZE}"
        f" --output-dir {shlex.quote(full_output_dir)}"
        f"{mbpp_arg}{taco_path_arg}{lexical_arg}"
    )
    return cmd, full_output_dir


def show_latest_metrics(output_subdir):
    run_root = Path(OUTPUT_DIR) / output_subdir
    if not run_root.exists():
        output_root = Path(OUTPUT_DIR)
        existing = sorted(path.name for path in output_root.iterdir() if path.is_dir()) if output_root.exists() else []
        raise FileNotFoundError(
            f"No run directory found for '{output_subdir}' under {run_root}. "
            f"Run the corresponding evaluation cell first. Existing output subfolders: {existing}"
        )
    run_dirs = sorted(path for path in run_root.iterdir() if path.is_dir())
    if not run_dirs:
        raise FileNotFoundError(f"{run_root} exists but has no timestamped runs yet. Run the evaluation cell first.")
    latest_run = run_dirs[-1]
    print("Latest run:", latest_run)
    return latest_run


def _slugify(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip("-")


def download_run_csvs(output_subdir, csv_names=None, experiment_label=None):
    from google.colab import files
    import pandas as pd

    csv_names = csv_names or ["metrics.csv"]
    run_dir = show_latest_metrics(output_subdir)
    metrics_path = run_dir / "metrics.csv"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Expected metrics.csv in {run_dir}")

    metrics = pd.read_csv(metrics_path)
    task_slug = _slugify(experiment_label or output_subdir)
    perturb_slug = _slugify("-".join(sorted(metrics["perturbation_tier"].dropna().astype(str).unique())))
    model_slug = f"models-{metrics['model_name'].nunique()}"
    config_slug = f"split-{_slugify(SPLIT)}__seed-{SPLIT_SEED}__{model_slug}"

    for csv_name in csv_names:
        source = run_dir / csv_name
        if not source.exists():
            print(f"Skipping missing file: {source}")
            continue
        destination_name = f"{task_slug}__{config_slug}__tiers-{perturb_slug}__{csv_name}"
        destination = run_dir / destination_name
        shutil.copy2(source, destination)
        print("Downloading:", destination)
        files.download(str(destination))


## 1. MBPP OOD Only

In [ ]:
cmd, mbpp_output_dir = build_eval_cmd(
    task="mbpp_ood",
    perturbation_tier="clean",
    output_subdir="mbpp_ood_only",
    lexical_map_path=LEXICAL_MAP_PATH,
)
print(cmd)
!cd /content/CS4248_ez_A && {cmd}

In [ ]:
import pandas as pd

latest_mbpp_run = show_latest_metrics("mbpp_ood_only")
mbpp_metrics = pd.read_csv(latest_mbpp_run / "metrics.csv")
mbpp_metrics

## 2. TACO Noisy Robustness Baselines

Change `NOISY_TIER` to `all`, `typo_light`, `typo_heavy`, `grammar_light`, `mixed_light`, or `mixed_heavy`.

In [ ]:
NOISY_TIER = "all"

In [ ]:
cmd, noisy_output_dir = build_eval_cmd(
    task="taco_robustness",
    perturbation_tier=NOISY_TIER,
    output_subdir="taco_noisy_baselines",
    lexical_map_path=LEXICAL_MAP_PATH,
)
print(cmd)
!cd /content/CS4248_ez_A && {cmd}

In [ ]:
latest_noisy_run = show_latest_metrics("taco_noisy_baselines")
noisy_metrics = pd.read_csv(latest_noisy_run / "metrics.csv")
noisy_metrics

## 3. TACO Keyword-Replacement Probes

Change `LEXICAL_TIER` to one of:
- `keyword_synonym`
- `keyword_neutralize`
- `keyword_swap_type`
- `identifier_mask`
- `structure_preserve_lexical_change`


In [ ]:
LEXICAL_TIER = "keyword_swap_type"

In [ ]:
cmd, lexical_output_dir = build_eval_cmd(
    task="taco_robustness",
    perturbation_tier=LEXICAL_TIER,
    output_subdir=f"taco_{LEXICAL_TIER}",
    lexical_map_path=LEXICAL_MAP_PATH,
)
print(cmd)
!cd /content/CS4248_ez_A && {cmd}

In [ ]:
latest_lexical_run = show_latest_metrics(f"taco_{LEXICAL_TIER}")
lexical_metrics = pd.read_csv(latest_lexical_run / "metrics.csv")
lexical_probe_results = pd.read_csv(latest_lexical_run / "lexical_probe_results.csv")
example_cases = pd.read_csv(latest_lexical_run / "example_cases.csv")
lexical_metrics

In [ ]:
lexical_probe_results.head(20)

In [ ]:
example_cases

## 4. Combined Run If You Want Everything Together

In [ ]:
COMBINED_TIER = "all"  # set to one lexical tier if you want a single combined OOD + lexical run instead

In [ ]:
cmd, combined_output_dir = build_eval_cmd(
    task="all",
    perturbation_tier=COMBINED_TIER,
    output_subdir="combined_run",
    lexical_map_path=LEXICAL_MAP_PATH,
)
print(cmd)
!cd /content/CS4248_ez_A && {cmd}

## 5. Plot Latest TACO Run

In [ ]:
RUN_SUBDIR_TO_PLOT = f"taco_{LEXICAL_TIER}"  # or "taco_noisy_baselines", "combined_run", etc.

In [ ]:
import matplotlib.pyplot as plt

latest_plot_run = show_latest_metrics(RUN_SUBDIR_TO_PLOT)
plot_metrics = pd.read_csv(latest_plot_run / "metrics.csv")

taco = plot_metrics[plot_metrics["task"] == "taco_robustness"].copy()
taco_clean = taco[taco["perturbation_tier"] == "clean"].copy()
taco_noisy = taco[taco["perturbation_tier"] != "clean"].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(taco_clean["model_name"], taco_clean["mrr"])
axes[0].set_title("TACO Clean MRR")
axes[0].tick_params(axis="x", rotation=35)

for model_name, frame in taco_noisy.groupby("model_name"):
    axes[1].plot(frame["perturbation_tier"], frame["delta_mrr_vs_clean"], marker="o", label=model_name)
axes[1].set_title("TACO Robustness Drop (MRR)")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend()

mbpp = plot_metrics[plot_metrics["task"] == "mbpp_ood"].copy()
if not mbpp.empty and not taco_clean.empty:
    compare = taco_clean[["model_name", "mrr"]].rename(columns={"mrr": "taco_clean_mrr"}).merge(
        mbpp[["model_name", "mrr"]].rename(columns={"mrr": "mbpp_ood_mrr"}),
        on="model_name",
        how="outer",
    )
    compare = compare.set_index("model_name")
    compare.plot(kind="bar", ax=axes[2])
    axes[2].set_title("MBPP OOD vs TACO Clean")
    axes[2].tick_params(axis="x", rotation=35)
else:
    axes[2].axis("off")
    axes[2].set_title("No MBPP OOD data in selected run")

plt.tight_layout()
plt.show()

In [ ]:
# Example downloads:
# download_run_csvs("mbpp_ood_only", ["metrics.csv"], experiment_label="mbpp_ood")
# download_run_csvs("taco_noisy_baselines", ["metrics.csv", "per_query_results.csv"], experiment_label="taco_noisy")
download_run_csvs(
    f"taco_{LEXICAL_TIER}",
    ["metrics.csv", "lexical_probe_results.csv", "example_cases.csv"],
    experiment_label=f"taco_{LEXICAL_TIER}",
)